# Owen EDA: Batch 3 wines

This notebook selects every review whose `variety` belongs to batch 3 in `wine_style_groups_final.csv`. All EDA below runs on that subset only.

The raw data is loaded and cleaned the same way as `CSV_download.py` (both Kaggle files, merged and de-duplicated), so the review counts match the `num_reviews` column in the style groups file.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BATCH = 3
DEDUPE_KEY = ['description', 'winery']

style_groups = pd.read_csv('wine_style_groups_final.csv')

In [2]:
def load_clean(path):
    """Read one raw Kaggle file and clean it like CSV_download.py."""
    df = pd.read_csv(path, index_col=0)
    text_cols = df.select_dtypes(include=['object', 'str']).columns
    df[text_cols] = df[text_cols].apply(lambda s: s.str.strip()).replace('', np.nan)
    df = df.dropna(subset=['description', 'points'])
    df = df[df['points'].between(80, 100)].copy()
    df.loc[df['price'] <= 0, 'price'] = np.nan
    return df.drop_duplicates().drop_duplicates(subset=DEDUPE_KEY)


# Same merge as the team database: 130k first, then the first 150k, de-duplicated.
all_reviews = (
    pd.concat([load_clean('data/winemag-data-130k-v2.csv'),
               load_clean('data/winemag-data_first150k.csv')], ignore_index=True)
    .drop_duplicates(subset=DEDUPE_KEY, keep='first')
    .reset_index(drop=True)
)
print(f'All reviews after merge: {len(all_reviews):,}')

All reviews after merge: 169,613


In [3]:
# Style groups assigned to batch 3.
style_groups.loc[style_groups['batch'].eq(BATCH), ['style_group', 'num_varieties', 'num_reviews', 'pct']]

,style_group,num_varieties,num_reviews,pct
13,Cabernet Sauvignon,2,13290,7.84
14,Sauvignon Blanc,3,7290,4.30
15,Merlot,1,4757,2.80
16,"Chenin, Semillon & Loire/SW France Whites",32,3325,1.96
17,Nebbiolo,1,3126,1.84
18,Austrian & German Whites,41,2109,1.24


In [4]:
# Collect the exact variety names assigned to batch 3.
batch_varieties = (
    style_groups.loc[style_groups['batch'].eq(BATCH), 'varieties']
    .str.split(';')
    .explode()
    .str.strip()
    .dropna()
    .unique()
)

# `variety` is the matching column in the Wine Enthusiast data.
wines = all_reviews.loc[all_reviews['variety'].isin(batch_varieties)].copy()

expected = style_groups.loc[style_groups['batch'].eq(BATCH), 'num_reviews'].sum()
print(f'Batch {BATCH} varieties: {len(batch_varieties):,}')
print(f'Batch {BATCH} wine reviews: {len(wines):,} (style groups file says {expected:,})')
wines.head()

Batch 3 varieties: 80
Batch 3 wine reviews: 33,897 (style groups file says 33,897)


,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
10,US,"Soft, supple plum envelopes an oaky structure ...",Mountain Cuvée,87,19.0,California,Napa Valley,Napa,Virginie Boone,@vboone,Kirkland Signature 2011 Mountain Cuvée Caberne...,Cabernet Sauvignon,Kirkland Signature
12,US,"Slightly reduced, this wine offers a chalky, t...",NaN,87,34.0,California,Alexander Valley,Sonoma,Virginie Boone,@vboone,Louis M. Martini 2012 Cabernet Sauvignon (Alex...,Cabernet Sauvignon,Louis M. Martini
23,US,This wine from the Geneseo district offers aro...,Signature Selection,87,22.0,California,Paso Robles,Central Coast,Matt Kettmann,@mattkettmann,Bianchi 2011 Signature Selection Merlot (Paso ...,Merlot,Bianchi
29,US,Clarksburg is becoming a haven for Chenin Blan...,NaN,86,16.0,California,Clarksburg,Central Valley,Virginie Boone,@vboone,Clarksburg Wine Company 2010 Chenin Blanc (Cla...,Chenin Blanc,Clarksburg Wine Company
34,US,"This shows a tart, green gooseberry flavor tha...",NaN,86,20.0,California,Sonoma Valley,Sonoma,NaN,NaN,Envolve 2011 Sauvignon Blanc (Sonoma Valley),Sauvignon Blanc,Envolve


## EDA